# Übersetzung NL → DE — Runner

Code kommt per `git` in die VM, gearbeitet wird im Drive-Projektordner.
Jeder fertige Chunk liegt sofort dauerhaft in Drive.

**Ein Abbruch ist ein Nicht-Ereignis.** Trennt sich die VM, genügt ein
erneuter Klick auf Zelle 2 — der Resume zählt die Dateien in `teile/`
und setzt am nächsten offenen Chunk fort.

Reihenfolge beim ersten Start einer Sitzung: **0 → (1) → 2**.
Danach genügt Zelle 2. Zelle 1 nur im Sheets-Betrieb.

Secrets im Colab-Reiter „Secrets" hinterlegen und dieser Sitzung
Zugriff geben: `ANTHROPIC_API_KEY` und `GoogleKI`.

**Für ein neues Buch wird nur `PROJEKT` in Zelle 0 geändert.** Alles
Weitere steht in der `projekt.json` im Drive-Ordner; die entsteht beim
Erstlauf aus `projekt_vorlage.json` und wird nie überschrieben.

## 0 · Einstellungen und Code

Die **einzige** Zelle, die für ein neues Buch angefasst wird: `PROJEKT`
auf den Drive-Ordner mit der `input.txt` zeigen lassen.

Holt den Code, wirft alte Modulimporte weg und nennt den Kopf-Commit.
Steht dort nicht, was du erwartest, ist der Merge noch nicht durch —
dann nützt kein zweiter Versuch weiter unten.

In [ ]:
PROJEKT = "/content/drive/MyDrive/uebersetzung/1919"

REPO = "https://github.com/iinoox-ai/Claude-Code-Translate-.git"
CODE = "/content/Claude-Code-Translate-"

import os, subprocess, sys

def _git(*a):
    return subprocess.run(["git", "-C", CODE, *a],
                          capture_output=True, text=True)

if os.path.isdir(CODE):
    _git("fetch", "origin", "main")
    p = _git("pull", "--ff-only")
    if p.returncode != 0:
        print("git pull FEHLGESCHLAGEN:",
              (p.stderr or p.stdout).strip()[:300], "\n")
else:
    subprocess.run(["git", "clone", REPO, CODE], check=True)

# Frisch geholter Code nuetzt nichts, solange der Kernel die alten Module
# festhaelt: "import" ist dann ein No-op.
for _n, _m in list(sys.modules.items()):
    if (getattr(_m, "__file__", None) or "").startswith(CODE):
        del sys.modules[_n]
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import colab_start

print(_git("log", "--oneline", "-1").stdout.strip())
_hinten = _git("rev-list", "--count", "HEAD..origin/main").stdout.strip()
if _hinten not in ("", "0"):
    print(f"ACHTUNG: {_hinten} Commits hinter origin/main — "
          f"der Code in dieser VM ist ALT.")
else:
    print("aktuell mit origin/main")
print("Projekt:", PROJEKT)

## 1 · Bei Google anmelden — nur im Sheets-Betrieb

**Einmal je Colab-Sitzung**, und zwar hier in einer Zelle: Colab führt die
Anmeldung über den Kernel-Kanal zur Oberfläche, und den gibt es in einem
Unterprozess nicht.

Ohne diesen Schritt bricht der Preflight ab mit

```
Keine Google-Anmeldung — die ID und die Freigabe sind nicht das Problem.
```

Sagt die Ausgabe „Die Anmeldung gilt nur in dieser Zelle", laufen die
Sheets-Aufrufe über `colab_start.sync_im_kernel()` statt über
`colab_start.lauf(...)`.

Ohne `sheets_id` in der `projekt.json` überspringst du diese Zelle — dann
werden die Referenzdaten direkt als JSON gepflegt.

In [ ]:
colab_start.sheets_anmelden(code=CODE)

## 2 · Vorbereiten und laufen

Mountet Drive, zieht den Code nach, lädt die Secrets, wechselt in den
Projektordner und startet den Lauf im Vordergrund. Die laufende
Fortschrittsausgabe hält die Sitzung nebenbei wach — sie ist kein Beiwerk.

Beim Erstlauf entsteht dabei die `projekt.json` im Projektordner aus
`projekt_vorlage.json`. **Ab dann gilt die Datei im Drive-Ordner**;
`sheets_id`, `rahmen_marker` und alles Weitere werden dort geändert, nicht
im Repo.

Meldet diese Zelle abweichende technische Einstellungen, erst Zelle 4
ausführen und danach hierher zurück.

In [ ]:
import subprocess, sys
subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
for _n, _m in list(sys.modules.items()):
    if (getattr(_m, "__file__", None) or "").startswith(CODE):
        del sys.modules[_n]
import colab_start

colab_start.vorbereiten(PROJEKT, code=CODE)
colab_start.lauf("pipeline.py", "run", code=CODE)

## 3 · Eine Pause freigeben

Der Lauf hält zweimal von selbst an: nach der Vorbereitung
(`PAUSE_review`) und nach dem Testlauf (`PAUSE_pruefung`). Was dort zu
lesen und zu entscheiden ist, steht in `NEUES_BUCH.md`.

Diese Zelle hakt **genau eine** offene Pause ab und läuft weiter. Einen
fehlgeschlagenen Schritt hakt sie nie ab — dafür ist `reset` da.

In [ ]:
colab_start.lauf("pipeline.py", "weiter", code=CODE)

## 4 · Technische Einstellungen abgleichen

Die `projekt.json` im Drive-Ordner wird nie überschrieben — sie trägt die
kalibrierten Prüfgrenzen und die Entscheidungen dieses Buchs. Modellnamen
und Effort gehören dagegen zum Code und müssen nachgezogen werden, wenn
sich in `projekt_vorlage.json` etwas geändert hat.

Was dieses Buch für sich beansprucht (`technik_ausnahmen`), bleibt dabei
stehen und wird nur gemeldet.

In [ ]:
# ohne "--uebernehmen" nur anzeigen
colab_start.lauf("pipeline.py", "technik", "--uebernehmen", code=CODE)

## 5 · Verifikation — einmalig vor dem ersten Volllauf

Prüft die Schreibsemantik des Drive-Mounts, pingt jedes konfigurierte
Modell, übersetzt einen Kurz-Chunk je Anbieter mit Ausweis der
Token-Usage, belegt die Sampling-Doktrin an der echten API, prüft ob der
Ablehnungsrückfall freigeschaltet ist, und gleicht die Google-Tarife gegen
die Preisseite ab.

Kostet wenige Cent. **Nach Zelle 4 ausführen** — sonst prüft sie womöglich
Modellnamen, die das Repo längst korrigiert hat.

In [ ]:
colab_start.lauf("verifikation.py", code=CODE)

## 6 · Nachsehen

Colab arbeitet Zellen nacheinander ab: Solange Zelle 2 läuft, wird eine
Zelle hier nur eingereiht, nicht ausgeführt. Den Chunkstand während eines
Laufs liest man deshalb an der Ausgabe von Zelle 2 ab.

In [ ]:
colab_start.lauf("pipeline.py", "status", code=CODE)

In [ ]:
# Modell und Tiefe je Rolle, neben der Empfehlung mit Begruendung
colab_start.lauf("pipeline.py", "modelle", code=CODE)

In [ ]:
# Plan fuer den Stapelbetrieb: Ketten, Wellen, zusaetzliche Naehte.
# Der Handel dahinter steht in NEUES_BUCH.md, Schritt 11a.
colab_start.lauf("pipeline.py", "wellen", code=CODE)

## 7 · Schlüssel-Diagnose

Nur im Fehlerfall. Wenn der Preflight meldet, ein Anbieter weise den
Schlüssel zurück, beantwortet diese Zelle drei Fragen, die die Meldung des
Anbieters offen lässt:

1. Steht in der **Umgebung** etwas anderes als im **Colab-Secret**? Die
   Umgebung hat Vorrang — ein einmal falsch hineingeratener Wert bleibt die
   ganze Sitzung und verdeckt das Secret.
2. Sieht ein **Unterprozess** dasselbe? Alle Schritte laufen als Unterprozess.
3. Antwortet der Anbieter, wenn der Wert **direkt aus dem Secret** kommt, an
   unserem Lesepfad vorbei? Das trennt „der Schlüssel ist ungültig" von
   „unser Code verdirbt ihn unterwegs".

Der Schlüssel selbst wird nirgends ausgegeben — nur Länge, Präfix und ein
Fingerabdruck, mit dem sich zwei Werte vergleichen lassen, ohne einen davon
zu sehen.

In [ ]:
colab_start.schluessel_diagnose(code=CODE)

## 8 · Absätze richten — nur wenn der Preflight es sagt

Meldet der Preflight `Nur N Absaetze erkannt` und nennt als Befund
**EINE ZEILE JE ABSATZ**, dann trennt die Datei ihre Absätze mit einem
einfachen Umbruch statt mit einer Leerzeile. Diese Zelle verdoppelt jeden
Umbruch und legt vorher `input.txt.bak` mit dem Original an.

Sie tut **nichts**, wenn der Befund ein anderer ist. Bei einem mitten im Satz
umbrochenen Text (PDF-Extraktion) würde die Verdopplung aus jeder halben Zeile
einen Absatz machen — dort hilft nur ein Export, der Absätze erhält.

Danach Zelle 2 erneut.

In [ ]:
colab_start.lauf("preflight.py", "--umbrueche", code=CODE)